In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

Two big limitations of LLMs are 

1) that they only "know" the information that they were trained on
2) that they have limited input context windows.

A way to address both of these limitations is to use a technique called Retrieval Augmented Generation, or RAG. A RAG system has three stages:

* Indexing
* Retrieval
* Generation

Indexing happens ahead of time, and allows you to quickly look up relevant information at query-time. When a query comes in, you retrieve relevant documents, combine them with your instructions and the user's query, and have the LLM generate a tailored answer in natural language using the supplied information. This allows you to provide information that the model hasn't seen before, such as product-specific knowledge or live weather updates.

In this notebook you will use the Gemini API to create a vector database, retrieve answers to questions from the database and generate a final answer. You will use Chroma, an open-source vector database. With Chroma, you can store embeddings alongside metadata, embed documents and queries, and search your documents.

## **⚙️Setup**
First, install ChromaDB and the Gemini API Python SDK.

In [2]:
!pip uninstall -qqy jupyterlab kfp  # Remove unused conflicting packages
!pip install -qU "google-genai==1.7.0" "chromadb==0.6.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.7/144.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.7/198.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [3]:
from google import genai
from google.genai import types

from IPython.display import Markdown

genai.__version__

'1.7.0'

## **🔑Set up your API key**
To run the following cell, your API key must be stored it in a Kaggle secret named GOOGLE_API_KEY.

To make the key available through Kaggle secrets, choose Secrets from the Add-ons menu and follow the instructions to add your key or enable it for this notebook.

In [4]:
from kaggle_secrets import UserSecretsClient

GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")

## 🔍Explore available models

We will be using the embedContent API method to calculate embeddings. Find a model that supports it through the models.list endpoint. You can also find more information about the embedding models on the models page.

**text-embedding-004** is the most generally-available embedding model, so we will use it for this exercise. You can try with other models too.

In [5]:
client = genai.Client(api_key=GOOGLE_API_KEY)

for m in client.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

models/gemini-embedding-001


## **📋 Data**
Here is a small set of documents you will use to create an embedding database.

In [6]:
DOCUMENT1 = """BONES
Clubfoot -- Nux v., Phos., Strych.
Cold feeling -- Zinc. m.
Condyles, epiphyses, swollen -- Conchiolin, Rhus t.
Sutures, affected -- Calc. c., Calc. p.
Cranial bones, thin, soft -- Calc. c., Calc. p.
Crooking -- Am. c., Calc. c., Calc. p., Iod., Sil.
Development, tardy -- Calc. c., Calc. fl., Calc. p., Sil.
Enlargement (acromegaly) -- Pituit. ext., Thyr.
Exostoses -- Arg. m., Aur., Calc. c., Calc. fl., Fluor. ac., Hekla, Kali bich., Kali iod., Lapis alb., Malandr., Merc. c., Merc. phos., Merc. s., Mez., Phos., Plumb. ac., Ruta, Sil., Still., Sul., Zinc. m.
Fractures
Shock -- Acon., Arn.
Slow union -- Calc. p., Calend., Iod., Mang. ac., Mez., Phos. ac., Ruta, Sil., Symphyt., Thyr.
Inflammation (osteitis) -- Asaf., Aur. iod., Aur., Conchiol., Hekla, Hep., Iod., Kali iod., Merc. s., Mez., Nit. ac., Phos. ac., Phos., Staph., Still., Stront. c.
Chronic (osteitis deformans) -- Aur., Calc. p., Hekla, Nit. ac.
Osteomyelitis -- Acon., Chin. s., Gun powder, Phos."""

DOCUMENT2 = """
CANCER -- Acet. ac., Ananth., Ant. chlor., Apis, Ars., Ars. br., Ars. iod., Aster., Aur. ars., Aur. m. n., Bapt., Bism., Brom., Calc. c., Calc. iod., Calc. ox., Calend., Carb. ac., Carbo an., Carbon s., Carcinos., Choline, Cic., Cinnam., Cistus., Condur., Con., Cupr. ac., Eosin., Euphorb., Form. ac., Formica, Fullgo, Galium ap., Guaco, Graph., Ham., Hoang n., Hydr., Iod., Kali ars., Kali cy., Kali iod., Kreos., Lach., Lapis alb., Lyc., Maland., Med., Phos., Phyt., Radium, Rumex ac., Sang., Semperv. t., Scirrhin., Sedum rep., Sep., Sil., Symphyt., Sul., Taxus, Thuya.
Antrum [of] -- Aur., Symphyt.
Bone [of] -- Aur. iod., Phos., Symphyt.
Bowel, lower [of] -- Ruta.
Breast [of] (See Female Sexual System.) -- Ars. iod., Bar. iod., Brom., Bufo, Carbo an., Carcinos., Condur., Con., Form. ac., Graph., Hydr., Phyt., Plumb. iod., Nat. cacodyl., Scirrhin.
Cæcum [of] -- Ornithog.
Glandular structures [of] -- Hoang nan.
Omentum [of] -- Lob. erin.
Stomach [of] (See Stomach.) -- Acet. ac., Ars., Bism., Cadm. s., Condur., Form. ac., Hydr., Kreos., Ornithog., Phos., Sec.
Uterus [of] (See Female Sexual System.) -- Aur. m. n., Carbo an., Carcinos., Fuligo, Hydr., Iod., Lapis alb., Nat. cacodyl., Nit. ac., Sec.
To relieve pains -- Alveloz., Apis, Anthrac., Ars., Aster., Bry., Calc. ac., Calc. c., Calc. ox., Carcinos., Ced., Cinnam., Condur., Con., Echin., Euphorb., Hydr., Mag. p., Morph., Op., Ova t., Phos. ac., Sil.
"""
DOCUMENT3 = "CARTILAGES (perichondritis), Inflammation -- Arg. m., Bell., Cham., Cim., Oleand., Plumb., Ruta., Pains -- Arg. m., Ruta., Ulceration -- Merc. c."

documents = [DOCUMENT1, DOCUMENT2, DOCUMENT3]

## **📝Creating the embedding database with ChromaDB**
Create a custom function to generate embeddings with the Gemini API. In this task, we are implementing a retrieval system, so the **task_type** for generating the document embeddings is **retrieval_document**. Later, we will use retrieval_query for the query embeddings.

Key words: Documents are the items that are in the database. They are inserted first, and later retrieved. Queries are the textual search terms and can be simple keywords or textual descriptions of the desired documents.

In [7]:
from chromadb import Documents, EmbeddingFunction, Embeddings
from google.api_core import retry

from google.genai import types


# Define a helper to retry when per-minute quota is reached.
is_retriable = lambda e: (isinstance(e, genai.errors.APIError) and e.code in {429, 503})


class GeminiEmbeddingFunction(EmbeddingFunction):
    # Specify whether to generate embeddings for documents, or queries
    document_mode = True

    @retry.Retry(predicate=is_retriable)
    def __call__(self, input: Documents) -> Embeddings:
        if self.document_mode:
            embedding_task = "retrieval_document"
        else:
            embedding_task = "retrieval_query"

        response = client.models.embed_content(
            model="models/gemini-embedding-001",
            contents=input,
            config=types.EmbedContentConfig(
                task_type=embedding_task,
            ),
        )
        return [e.values for e in response.embeddings]


Now create a Chroma database client that uses the **GeminiEmbeddingFunction** and populate the database with the documents you defined above.

In [8]:
import chromadb

DB_NAME = "medicinesdb"

embed_fn = GeminiEmbeddingFunction()
embed_fn.document_mode = True

chroma_client = chromadb.Client()
db = chroma_client.get_or_create_collection(name=DB_NAME, embedding_function=embed_fn)

db.add(documents=documents, ids=[str(i) for i in range(len(documents))])

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Confirm that the data was inserted by looking at the database.

In [9]:
db.count()
# You can peek at the data too.
db.peek(1)

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


{'ids': ['0'],
 'embeddings': array([[ 0.00721748, -0.02262152,  0.01908747, ...,  0.00844967,
          0.00402329, -0.01565993]]),
 'documents': ['BONES\nClubfoot -- Nux v., Phos., Strych.\nCold feeling -- Zinc. m.\nCondyles, epiphyses, swollen -- Conchiolin, Rhus t.\nSutures, affected -- Calc. c., Calc. p.\nCranial bones, thin, soft -- Calc. c., Calc. p.\nCrooking -- Am. c., Calc. c., Calc. p., Iod., Sil.\nDevelopment, tardy -- Calc. c., Calc. fl., Calc. p., Sil.\nEnlargement (acromegaly) -- Pituit. ext., Thyr.\nExostoses -- Arg. m., Aur., Calc. c., Calc. fl., Fluor. ac., Hekla, Kali bich., Kali iod., Lapis alb., Malandr., Merc. c., Merc. phos., Merc. s., Mez., Phos., Plumb. ac., Ruta, Sil., Still., Sul., Zinc. m.\nFractures\nShock -- Acon., Arn.\nSlow union -- Calc. p., Calend., Iod., Mang. ac., Mez., Phos. ac., Ruta, Sil., Symphyt., Thyr.\nInflammation (osteitis) -- Asaf., Aur. iod., Aur., Conchiol., Hekla, Hep., Iod., Kali iod., Merc. s., Mez., Nit. ac., Phos. ac., Phos., Staph.,

## 📋🔍Retrieval: Find relevant documents

To search the Chroma database, call the query method. Note that you also switch to the **retrieval_query** mode of embedding generation.

In [10]:
# Switch to query mode when generating embeddings.
embed_fn.document_mode = False

# Search the Chroma DB using the specified query.
query = "Medicines for BONES for Exostoses"

result = db.query(query_texts=[query], n_results=1)
[all_passages] = result["documents"]

Markdown(all_passages[0])

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


BONES
Clubfoot -- Nux v., Phos., Strych.
Cold feeling -- Zinc. m.
Condyles, epiphyses, swollen -- Conchiolin, Rhus t.
Sutures, affected -- Calc. c., Calc. p.
Cranial bones, thin, soft -- Calc. c., Calc. p.
Crooking -- Am. c., Calc. c., Calc. p., Iod., Sil.
Development, tardy -- Calc. c., Calc. fl., Calc. p., Sil.
Enlargement (acromegaly) -- Pituit. ext., Thyr.
Exostoses -- Arg. m., Aur., Calc. c., Calc. fl., Fluor. ac., Hekla, Kali bich., Kali iod., Lapis alb., Malandr., Merc. c., Merc. phos., Merc. s., Mez., Phos., Plumb. ac., Ruta, Sil., Still., Sul., Zinc. m.
Fractures
Shock -- Acon., Arn.
Slow union -- Calc. p., Calend., Iod., Mang. ac., Mez., Phos. ac., Ruta, Sil., Symphyt., Thyr.
Inflammation (osteitis) -- Asaf., Aur. iod., Aur., Conchiol., Hekla, Hep., Iod., Kali iod., Merc. s., Mez., Nit. ac., Phos. ac., Phos., Staph., Still., Stront. c.
Chronic (osteitis deformans) -- Aur., Calc. p., Hekla, Nit. ac.
Osteomyelitis -- Acon., Chin. s., Gun powder, Phos.

## **📺Augmented generation: Answer the question**

Now that you have found a relevant passage from the set of documents (the retrieval step), you can now assemble a generation prompt to have the Gemini API generate a final answer. 

Note that in this example only a single passage was retrieved. In practice, especially when the size of your underlying data is large, you will want to retrieve more than one result and let the Gemini model determine what passages are relevant in answering the question. For this reason it's OK if some retrieved passages are not directly related to the question - this generation step should ignore them.

In [11]:
query_oneline = query.replace("\n", " ")

# This prompt is where you can specify any guidance on tone, or what topics the model should stick to, or avoid.
prompt = f"""You are a helpful and informative bot that answers questions using text from the reference passage included below. 
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. 
However, you are talking to a non-technical audience, so be sure to break down complicated concepts and 
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: {query_oneline}
"""

# Add the retrieved documents to the prompt.
for passage in all_passages:
    passage_oneline = passage.replace("\n", " ")
    prompt += f"PASSAGE: {passage_oneline}\n"

print(prompt)

You are a helpful and informative bot that answers questions using text from the reference passage included below. 
Be sure to respond in a complete sentence, being comprehensive, including all relevant background information. 
However, you are talking to a non-technical audience, so be sure to break down complicated concepts and 
strike a friendly and converstional tone. If the passage is irrelevant to the answer, you may ignore it.

QUESTION: Medicines for BONES for Exostoses
PASSAGE: BONES Clubfoot -- Nux v., Phos., Strych. Cold feeling -- Zinc. m. Condyles, epiphyses, swollen -- Conchiolin, Rhus t. Sutures, affected -- Calc. c., Calc. p. Cranial bones, thin, soft -- Calc. c., Calc. p. Crooking -- Am. c., Calc. c., Calc. p., Iod., Sil. Development, tardy -- Calc. c., Calc. fl., Calc. p., Sil. Enlargement (acromegaly) -- Pituit. ext., Thyr. Exostoses -- Arg. m., Aur., Calc. c., Calc. fl., Fluor. ac., Hekla, Kali bich., Kali iod., Lapis alb., Malandr., Merc. c., Merc. phos., Merc. s.,

Now use the **generate_content** method to to generate an answer to the question.

In [12]:
answer = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt)

Markdown(answer.text)

For something called exostoses, which are essentially bone growths, the passage suggests a variety of remedies that might be considered to help, including Arg. m., Aur., Calc. c., Calc. fl., Fluor. ac., Hekla, Kali bich., Kali iod., Lapis alb., Malandr., Merc. c., Merc. phos., Merc. s., Mez., Phos., Plumb. ac., Ruta, Sil., Still., Sul., and Zinc. m.